# Can concepts predict temporal performance deterioration?

At the end of year **t**, we know the fitted model's concept measurements and observed performance through **t**. We predict the change to **t+1**. Positive deterioration means worse performance: current minus next for frozen-cutoff F1 and average precision; next minus current for Brier score.

The mortality cutoff is chosen on reference-year validation patients and stays fixed. Oracle cutoffs are descriptive only. Brier score measures overall probability error, not calibration alone.

This notebook only reads validated, completed artifacts. Run `run_temporal_concept_forecasting.py` to produce them; executing this notebook cannot launch experiments.

The report may cover a selected experiment scope instead of the full historical grid. The scope, reuse provenance, forecast-eligibility exclusions and limitations are shown alongside the results below.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from artifact_storage import file_sha256, read_artifact
from temporal_concept_forecasting import checked_manifest, load_forecasting

if 'REPORT_MANIFEST' not in globals():
    pointer_path = Path('stats/temporal_concept_forecasting/report_manifest.json')
    if not pointer_path.exists():
        raise FileNotFoundError('No report artifacts yet. Run run_temporal_concept_forecasting.py first.')
    pointer = json.loads(pointer_path.read_text())
    REPORT_MANIFEST = pointer['path']
    assert file_sha256(REPORT_MANIFEST) == pointer['sha256']
report_path = Path(REPORT_MANIFEST)
report = checked_manifest(report_path)
runs = {}
for source in report['forecast_manifests']:
    assert file_sha256(source['path']) == source['sha256']
    data = load_forecasting(source['path'])
    runs[data['manifest']['system']] = data
print('Systems:', ', '.join(runs))
print('Stage B complete:', report['stage_b_complete'])
if not report['stage_b_complete']:
    display(Markdown('**Training-window analysis is not complete. No conclusion about its concept predictability is available yet.**'))
comparisons = pd.DataFrame([r for data in runs.values() for r in data['comparisons']])
folds = pd.DataFrame([r for data in runs.values() for r in data['folds']])
performance = pd.DataFrame([r for data in runs.values() for r in data['performance']])

## Experiment scope and provenance

Requested scope is what the window run was asked to cover; completed jobs come from the window manifest the report was built from. The original system is shown only as context. A reused fit is a validated copy of a result from an earlier reviewed run, recorded for provenance; nothing is refitted by this notebook.

In [ ]:
def year_span(years):
    years = sorted(years)
    return f'{years[0]}-{years[-1]}' if years == list(range(years[0], years[-1] + 1)) else ', '.join(map(str, years))

def show_wide(frame):
    with pd.option_context('display.max_colwidth', None, 'display.width', 200):
        display(frame)

scope = report.get('scope')
window_info = report.get('window_manifest')
REUSE_COLUMNS = ['reuse_basis', 'reused_from', 'source_run_identity', 'reused_manifest_sha256']
completed_fits = pd.DataFrame()
if window_info is None:
    print('No window manifest is recorded in this report: completed jobs and reuse provenance cannot be shown.')
elif not Path(window_info['path']).exists():
    print(f"Window manifest not found at {window_info['path']}: completed jobs and reuse provenance cannot be shown.")
else:
    window_path = Path(window_info['path'])
    assert file_sha256(window_path) == window_info['sha256']
    window_manifest = checked_manifest(window_path)
    completed_fits = pd.DataFrame(read_artifact(window_path.parent, window_manifest['artifacts']['aliases']))
    records = []
    for row in completed_fits.to_dict('records'):
        completed_path = window_path.parent / row['completed_manifest']
        assert file_sha256(completed_path) == row['completed_manifest_sha256']
        records.append(json.loads(completed_path.read_text()))
    for column in REUSE_COLUMNS:
        completed_fits[column] = [record.get(column) for record in records]
    completed_fits['origin'] = ['reused' if basis else 'computed' for basis in completed_fits['reuse_basis']]

if scope is None:
    print('Legacy report: no selected scope was recorded. Completeness is judged only by the Stage B flag above.')
else:
    print('Requested scope')
    print('  Reference years:', year_span(scope['reference_years']))
    print('  Windows:', ', '.join(scope['windows']))
    print('  Patient split seeds:', ', '.join(map(str, scope['patient_split_seeds'])))
    print('  Logical jobs:', scope['logical_jobs'])
    print('  Distinct fits:', scope['distinct_fits'])
    if not completed_fits.empty:
        requested = {tuple(job) for job in scope['selection']}
        in_scope = completed_fits[[(r, s, w) in requested for r, s, w in
                                   zip(completed_fits['reference_year'], completed_fits['patient_split_seed'], completed_fits['window'])]]
        print('Completed scope')
        print('  Completed jobs:', len(in_scope), 'of', len(requested), 'requested')
        print('  Completed distinct fits:', in_scope['fit_identity'].nunique())
        shared = in_scope.groupby('fit_identity').filter(lambda group: len(group) > 1)
        for fit, group in shared.groupby('fit_identity'):
            jobs = ', '.join(f'{r}/{w}' for r, w in zip(group['reference_year'], group['window']))
            print(f'  Jobs sharing one fit (identical training membership): {jobs}')
        completed_fits = in_scope
        show_wide(completed_fits.pivot(index='reference_year', columns='window', values='origin'))

contextual = report.get('contextual_systems', ['original'] if 'original' in runs else [])
print('Contextual Stage A systems:', ', '.join(contextual) or 'none')
print('These are frozen original artifacts: not a new fit, not part of the selected scope, and not evidence of patient-split replication.')

if not completed_fits.empty:
    reused = completed_fits[completed_fits['origin'] == 'reused']
    if reused.empty:
        print('Reused completed fits: none recorded; every completed fit in this report was computed by its own run.')
    else:
        print(f"Reused completed fits: {reused['fit_identity'].nunique()} distinct fits across {len(reused)} of {len(completed_fits)} logical jobs")
        show_wide(reused[['reference_year', 'window', 'fit_identity'] + REUSE_COLUMNS])

## Primary result

Does history plus the five concept measurements forecast frozen-cutoff F1 deterioration more accurately than history alone? The primary population is all-comers, activation fraction 0.5, with forward evaluation.

An improvement above zero means lower forecasting error. Intervals resample whole reference years, after averaging split losses within forecast years. References reuse calendar years and some patients; these intervals do not establish generalization to independent populations.

In [ ]:
primary = comparisons[comparisons['primary'] == True] if not comparisons.empty else pd.DataFrame()
if primary.empty:
    print('Primary comparison is not estimable; inspect exclusions below.')
else:
    display(primary[['system','improvement','lower_95','upper_95','paired_rows','references']])
    for r in primary.to_dict('records'):
        supported = r['lower_95'] is not None and pd.notna(r['lower_95']) and r['lower_95'] > 0
        print(f"{r['system']}: {'evidence of lower forecasting error' if supported else 'no clear improvement established'} on {r['references']} held reference years.")

## Other outcomes and representations

Each outcome, cohort, activation fraction and representation is evaluated separately. History includes current performance, its previous change, and model age. Concepts add current values, previous-year changes and coverage. PCA, factor analysis, CRI normalization, scaling and DAE are fitted inside the training folds. DAE uses the prespecified architecture and 250 epochs, averaging predictions across seeds 42–44.

The TCAV extension is secondary; its missing measurements do not remove rows from the core analysis. Each comparison below uses identical forecast rows for the two models. History with maximal coverage is reported separately.

In [ ]:
if not comparisons.empty:
    selected = comparisons.query("design == 'forward' and cohort_view == 'all_comer' and activation_target == 0.5 and profile == 'core' and metric == 'mae'")
    display(selected[['system','outcome','model','baseline','improvement','lower_95','upper_95','paired_rows','references']])
    fig, ax = plt.subplots(figsize=(9, max(3, len(primary)*0.7)))
    if not primary.empty:
        p = primary.reset_index(drop=True)
        ax.axvline(0, color='gray', linewidth=1)
        ax.scatter(p['improvement'], range(len(p)))
        for i, r in p.iterrows():
            if pd.notna(r['lower_95']) and pd.notna(r['upper_95']):
                ax.plot([r['lower_95'], r['upper_95']], [i,i], color='C0')
        ax.set_yticks(range(len(p)), p['system'])
        ax.set_xlabel('Reduction in forecasting MAE versus history (positive is better)')
        ax.set_title('Primary forward comparison with 95% reference-cluster intervals')
    plt.tight_layout()
    plt.show()

## Coverage and timing audit

Missing concepts are exclusions, not zero robustness. Early forward folds may have too few earlier reference systems to fit a forecasting model. Inspect support before interpreting differences.

In [ ]:
if not folds.empty:
    audit = folds[folds['status'].isin(['non_estimable','excluded'])]
    display(audit.groupby(['system','status','reason'], dropna=False).size().rename('fold/model entries').reset_index())
    forward = folds[folds['design']=='forward']
    assert all(all(y < r['held_reference'] for y in r['train_references']) for r in forward.to_dict('records'))
    assert all(pd.isna(r['maximum_training_outcome_year']) or r['maximum_training_outcome_year'] <= r['origin'] for r in forward.to_dict('records'))
    print('Forward timing checks passed.')
summary = pd.DataFrame([r for data in runs.values() for r in data['summaries']])
if not summary.empty:
    display(summary.query("model in ['history','history_maximal','no_change'] and activation_target == 0.5 and profile == 'core'"))

## Forecast eligibility

A forecast needs the previous, current and future annual performance of the same frozen system. A reference year without all three has performance and concept records but no forecast rows; with follow-up only through the next year, that is the last reference year (2014 in the priority scope). Those origins are listed as exclusions, never as zero or failed forecasts.

In [ ]:
artifacts = report['artifacts']
if 'forecast_eligibility' not in artifacts:
    print('No forecast-eligibility tables were recorded in this report; excluded origins appear only in the fold audit above.')
else:
    eligibility = pd.DataFrame(read_artifact(report_path.parent, artifacts['forecast_eligibility']))
    exclusions = pd.DataFrame(read_artifact(report_path.parent, artifacts['forecast_exclusions']))
    counts = report.get('forecast_eligibility', {})
    print('Descriptive-only reference years:', counts.get('descriptive_only_references', []))
    print('Forecast rows:', counts.get('forecast_rows'), '| Exclusion rows:', counts.get('exclusion_rows'))
    show_wide(eligibility.groupby(['system', 'status', 'reason'], dropna=False).agg(
        references=('reference_year', 'nunique'), performance_rows=('performance_rows', 'sum'),
        measured_concept_rows=('measured_concept_rows', 'sum'), forecast_rows=('forecast_rows', 'sum')).reset_index())
    if exclusions.empty:
        print('No forecast origins were excluded.')
    else:
        exclusions['missing_performance_years'] = exclusions['missing_performance_years'].map(lambda years: ', '.join(map(str, years)) or '-')
        show_wide(exclusions.groupby(['system', 'reference_year', 'forecast_origin_year', 'status', 'reason', 'missing_performance_years'])
                  .size().rename('outcomes').reset_index())
        for text in exclusions['explanation'].unique():
            print('-', text)
        print('Excluded origins are not forecasts and are not zero deterioration, failed predictions or negative results.')

## Training windows: three separate questions

1. Are performance levels better?
2. Is subsequent deterioration smaller?
3. Do concepts forecast deterioration more accurately beyond history?

These are different claims. Comparisons use common reference-only as the baseline and require identical record membership. Factor identities stay local to each fitted system. Expanding training and concept discovery together does not identify the separate contribution of either change.

In a selected-scope report these comparisons cover only the selected windows and reference years; the original system stays contextual.

In [ ]:
window_comparisons = pd.DataFrame(read_artifact(report_path.parent, report['artifacts']['window_comparisons']))
if window_comparisons.empty:
    print('Matched training-window comparisons are not available.')
else:
    display(window_comparisons[window_comparisons['cohort_view']=='all_comer'])
    print('All differences are window minus common reference-only.')
    print('Performance: higher F1/AP and lower Brier are better. Deterioration: lower is better. Concept forecasting gain: higher is better.')
if not performance.empty:
    all_comer = performance[(performance['cohort_view']=='all_comer') & (performance['valid']==True)]
    trajectory = all_comer.groupby(['system','reference_year','temporal_distance'])['death_f1_at_frozen_threshold'].mean().groupby(['system','temporal_distance']).mean().unstack(0)
    trajectory.plot(marker='o', ylabel='Frozen-cutoff death F1', xlabel='Years since reference', title='Descriptive performance trajectories; reference support varies by distance', figsize=(9,4))
    plt.tight_layout()
    plt.show()

## Limitations

Read every comparison above together with these limits.

In [ ]:
if report.get('limitations') is None:
    print('No limitations were recorded in this report.')
for text in report.get('limitations') or []:
    print('-', text)
if scope is not None and len(scope['patient_split_seeds']) == 1:
    print('- SAE initialization seeds (42/43/44 in this protocol) vary the concept ensemble inside one patient split; '
          'they do not replace patient-split replication.')